In [ ]:
!pip -q install pandapower==2.14.6 cvxpy clarabel 2>&1 | tail -2
import torch, pandapower, cvxpy
print("torch", torch.__version__, "| pandapower", pandapower.__version__,
      "| cvxpy", cvxpy.__version__)


In [ ]:
!rm -rf /kaggle/working/safesac-repo
!git clone -q --branch claude/thesis-q1-journal-path-komv2c \
    https://github.com/Rifat137710/Public.git /kaggle/working/safesac-repo
%cd /kaggle/working/safesac-repo
!git rev-parse --short HEAD


In [ ]:
!python -m pytest tests/test_powerflow.py tests/test_projection.py -q 2>&1 | tail -5
!python scripts/projected_heuristics.py --episodes 5 2>&1 | grep -v Warning | tail -20


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

!python scripts/transfer_study.py \
    --seeds 0 1 2 3 4 \
    --episodes 200 \
    --eval-episodes 20 \
    --alpha 0.003 \
    --train-z 0.5 \
    --deploy-z 0.5 2.0 4.0 6.0 \
    --load 0.40 \
    --evs 30 \
    --out-dir /kaggle/working/results/transfer \
    2>&1 | grep -v "UserWarning\|warnings.warn"


In [ ]:
import json, shutil
from pathlib import Path

res = Path("/kaggle/working/results/transfer/transfer.json")
d = json.loads(res.read_text())
print("train Z", d["train_z_pct"], "| deploy", d["deploy_z_pct"],
      "| seeds", d["seeds"], "| fingerprint", d["train_fingerprint"])
for z, row in d["summary"].items():
    print(f"Z={z:>4}%  " + "  ".join(
        f"{a}: {r['viol'][0]:.4f}/{r['soc'][0]:.3f}" for a, r in row.items()))

shutil.make_archive("/kaggle/working/transfer_results", "zip",
                    "/kaggle/working/results")
print("download /kaggle/working/transfer_results.zip")
